Chroma DB Local workflow

Building a traditional Rag system using langchain, chromadb and embedding model (hugging face)

In [2]:
# libraries needed
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

#vectorstore
from langchain_community.vectorstores import Chroma

#other
import numpy as np
from typing import List

C:\Users\Dell\AppData\Local\Temp\ipykernel_10224\1148882588.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Building sample data documents

In [3]:
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine Learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.
    It focuses on identifying patterns and making predictions based on historical information.
    Machine learning models improve their performance as they are exposed to more data.
    Common types include supervised, unsupervised, and reinforcement learning.
    Applications range from recommendation systems and fraud detection to image recognition and forecasting.
    The quality of data plays a crucial role in model performance.
    Proper evaluation helps ensure reliable and accurate predictions.
    """,

    """
    Deep Learning Fundamentals

    Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple layers.
    These networks are inspired by the structure and function of the human brain.
    Deep learning excels at processing large volumes of unstructured data such as images, audio, and text.
    It automatically learns complex features without requiring extensive manual feature engineering.
    Popular architectures include Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs).
    Deep learning has powered advances in computer vision, speech recognition, and generative AI.
    High computational resources are often required for training deep models.
    """,

    """
    Natural Language Processing Fundamentals
    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.
    It combines concepts from linguistics, computer science, and machine learning.
    NLP systems process text and speech to extract meaning and perform useful tasks.
    Common applications include chatbots, language translation, sentiment analysis, and text summarization.
    Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.
    Transformers have significantly improved the performance of NLP systems.
    Effective NLP solutions require both quality data and appropriate language representations.
    """
]

In [14]:
# saving sample doc to txt files
for i,doc in enumerate(sample_docs):
    with open(f"data/text_files/doc_{i}.txt","w") as f:
        f.write(doc)

 LOADING DOCUMENTS
 

In [4]:
from langchain_community.document_loaders import DirectoryLoader

In [5]:
# loading all text files from the directory
loader = DirectoryLoader(
    path = 'data/text_files',
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
docs = loader.load()
print(f"No of docs loaded: {len(docs)}")
print(docs)

No of docs loaded: 3
[Document(metadata={'source': 'data\\text_files\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine Learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed.\n    It focuses on identifying patterns and making predictions based on historical information.\n    Machine learning models improve their performance as they are exposed to more data.\n    Common types include supervised, unsupervised, and reinforcement learning.\n    Applications range from recommendation systems and fraud detection to image recognition and forecasting.\n    The quality of data plays a crucial role in model performance.\n    Proper evaluation helps ensure reliable and accurate predictions.\n    '), Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='\n    Deep Learning Fundamentals\n\n    Deep Learning is a specialized subset of machine learning that uses artificial neural n

SPLITTING DOCUMENTS INTO CHUNKS

In [6]:
# initializing splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)
chunks = splitter.split_documents(docs)
print(f"No of chunks created: {len(chunks)}")


No of chunks created: 6


APPLYING EMBEDDING MODELS and STORING CHUNKS IN CHROMA DB

In [7]:
# initializing embedding model hugging face 
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# CREATING CHROMA DB VECTOR STORE
persist_dir = "./chroma_db"

# initializing chromadb with hugging face embeddings
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding =  HuggingFaceEmbeddings(),
    persist_directory =  persist_dir,
    collection_name = "rag_collection"
)
print(f"No of vectors created in vector store: {vectorstore._collection.count()}")
print(f"Persisted to: {persist_dir}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

No of vectors created in vector store: 6
Persisted to: ./chroma_db


TESTING SIMILARITY SEARCH

In [10]:
query = " What is NLP?"
similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs


[Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Natural Language Processing Fundamentals\n    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.\n    It combines concepts from linguistics, computer science, and machine learning.\n    NLP systems process text and speech to extract meaning and perform useful tasks.\n    Common applications include chatbots, language translation, sentiment analysis, and text summarization.\n    Modern NLP heavily relies on deep learning'),
 Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.\n    Transformers have significantly improved the performance of NLP systems.\n    Effective NLP solutions require both quality data and appropriate language representations.'),
 Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_cont

ADVANCED SIMILARITY SEARCH WITH SCORES

In [11]:
results_scores = vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Natural Language Processing Fundamentals\n    Natural Language Processing (NLP) is a field that enables computers to understand, interpret, and generate human language.\n    It combines concepts from linguistics, computer science, and machine learning.\n    NLP systems process text and speech to extract meaning and perform useful tasks.\n    Common applications include chatbots, language translation, sentiment analysis, and text summarization.\n    Modern NLP heavily relies on deep learning'),
  0.7757503390312195),
 (Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Modern NLP heavily relies on deep learning models and word embeddings to capture language patterns.\n    Transformers have significantly improved the performance of NLP systems.\n    Effective NLP solutions require both quality data and appropriate language representations.'),
  1.19545316696167),
 (Document(metadata={'sou